# 🔍 AML Threshold (Structuring) Detection Engine
**System:** Anti-Money Laundering Detection | Neo4j Aura Enterprise 5.27  
**Module Coverage:** Module 1 → Module 7

---

### Architecture
```
Module 1  : Seed Selection        (weighted score: frequency + amount + degree)
Module 2  : Seed Batching
Module 3  : Extract Full Transaction History (incoming AND outgoing)
Module 3A : Sub-threshold Detector (configurable window, not fixed 90–99%)
Module 3B : Frequency Burst Detector
Module 4  : Merge & Remove Duplicates
Module 5  : Feature Engineering    (any_laundering EXCLUDED entirely)
Module 6  : Risk Scoring           (any_laundering EXCLUDED — reweighted)
Module 7  : Validation & Evaluation (any_laundering used HERE ONLY)
           + Export ALL HIGH/CRITICAL alerts
```

### Design decisions from this iteration
1. **Seed selection is no longer frequency-only.** It's now a weighted composite
   of transaction frequency, total transaction count, and total amount —
   frequency-heavy but not frequency-exclusive.
2. **Full transaction history is extracted** — both incoming and outgoing —
   not just outgoing. Structuring can happen on either side of an account.
3. **Batching happens after seed selection**, exactly like the other modules.
4. **The sub-threshold window is fully configurable** (`THRESHOLD_LIMIT`,
   `LOWER_PCT`, `UPPER_PCT`) rather than a hardcoded 90–99% band.
5. **`any_laundering` is removed from Module 5 (features) and Module 6
   (scoring) entirely.** It only appears in Module 7, where it is used purely
   as ground truth for evaluating the unsupervised risk score — this keeps
   the detector honestly unsupervised rather than leaking the label into its
   own score.
6. **Module 7 exports every HIGH and CRITICAL alert**, not just a top-N
   sample — investigators need the full actionable list, not a preview.

---
## ⚙️ Setup
Mount Google Drive and install required packages.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install neo4j pandas scikit-learn --quiet

---
## 🔌 Connect to Neo4j
Establish the driver connection and define shared helpers used throughout
this notebook.

In [ ]:
import pandas as pd
import numpy as np
import time
import os
from datetime import datetime, timezone
from google.colab import userdata
from neo4j import GraphDatabase

URI      = userdata.get('NEO4J_URI').strip()
USERNAME = userdata.get('NEO4J_USERNAME').strip()
PASSWORD = userdata.get('NEO4J_PASSWORD').strip()

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))
driver.verify_connectivity()
print('✓ Connected to Neo4j AuraDB')


def to_epoch(ts):
    """Convert a Neo4j DateTime object or ISO string to a Unix timestamp."""
    if ts is None:
        return None
    if hasattr(ts, 'to_native'):
        return ts.to_native().replace(tzinfo=timezone.utc).timestamp()
    if isinstance(ts, str):
        return datetime.fromisoformat(ts).timestamp()
    return float(ts)


def run_query_on_batch(session, query, batch, label, **extra_params):
    """Execute one Cypher query for one seed batch, with isolated error handling."""
    try:
        result = session.run(query, seed_ids=batch, **extra_params)
        return result.data()
    except Exception as e:
        print(f'  [WARN] {label} | batch failed → {e}')
        return []


def make_batches(id_list, batch_size=200):
    return [id_list[i:i + batch_size] for i in range(0, len(id_list), batch_size)]


print('Shared helpers defined ✓')

---
## MODULE 1 — Seed Selection (Weighted Candidate Score)
---

### Cell 1 : Compute Account Statistics
Pulls transaction count, total amount, and the first/last transaction time for
every account in one pass — everything needed to compute frequency without a
second round-trip to Neo4j.

In [ ]:
print('Computing account statistics...')

ACCOUNT_STATS_QUERY = """
MATCH (a:Account)

OPTIONAL MATCH (a)-[out:TRANSACTION]->()
WITH
    a,
    count(out)                        AS out_tx_count,
    coalesce(sum(out.amount_paid), 0) AS total_out_amount,
    min(out.timestamp)                AS min_out_ts,
    max(out.timestamp)                AS max_out_ts

OPTIONAL MATCH ()-[inc:TRANSACTION]->(a)
WITH
    a, out_tx_count, total_out_amount, min_out_ts, max_out_ts,
    count(inc)                        AS in_tx_count,
    coalesce(sum(inc.amount_paid), 0) AS total_in_amount,
    min(inc.timestamp)                AS min_in_ts,
    max(inc.timestamp)                AS max_in_ts

RETURN
    a.account_id AS account_id,
    out_tx_count,
    in_tx_count,
    (out_tx_count + in_tx_count)                     AS total_tx_count,
    total_out_amount,
    total_in_amount,
    (total_out_amount + total_in_amount)              AS total_transaction_amount,
    min_out_ts, max_out_ts, min_in_ts, max_in_ts
"""

with driver.session() as session:
    account_stats = session.run(ACCOUNT_STATS_QUERY).data()

account_df = pd.DataFrame(account_stats)
print(f'Accounts analysed : {len(account_df)}')
account_df.head()

### Cell 2 : Compute Transaction Frequency
Frequency = total transactions / active days. Active days is derived from the
earliest and latest transaction timestamp across BOTH directions — this is why
we pulled `min_out_ts/max_out_ts/min_in_ts/max_in_ts` above rather than just one side.

In [ ]:
def compute_active_days(row):
    """Span (in days) between an account's earliest and latest transaction,
    across both incoming and outgoing activity. Floored at 1 day to avoid
    division-by-zero for accounts with all activity in a single instant."""
    timestamps = [row['min_out_ts'], row['max_out_ts'], row['min_in_ts'], row['max_in_ts']]
    epochs = [to_epoch(t) for t in timestamps if t is not None]

    if len(epochs) < 2:
        return 1.0

    span_days = (max(epochs) - min(epochs)) / 86400.0
    return max(span_days, 1.0)


account_df['active_days'] = account_df.apply(compute_active_days, axis=1)
account_df['tx_frequency'] = account_df['total_tx_count'] / account_df['active_days']

print('Transaction frequency computed ✓')
account_df[['account_id', 'total_tx_count', 'active_days', 'tx_frequency']].head()

### Cell 3 : Weighted Candidate Seed Score
This REPLACES frequency-only ranking. The score is a weighted blend of three
normalised signals — frequency-heavy (structuring is fundamentally repetitive
behaviour) but not frequency-exclusive, so accounts with fewer but larger
sub-threshold transactions aren't missed.

In [ ]:
def normalize(series):
    if series.max() == series.min():
        return pd.Series([0.5] * len(series), index=series.index)
    return (series - series.min()) / (series.max() - series.min())


account_df['score_frequency'] = normalize(account_df['tx_frequency'])
account_df['score_tx_count']  = normalize(account_df['total_tx_count'])
account_df['score_amount']    = normalize(account_df['total_transaction_amount'])

# Weighted composite: frequency-heavy, but amount and raw count both contribute
account_df['candidate_seed_score'] = (
      0.50 * account_df['score_frequency']
    + 0.30 * account_df['score_tx_count']
    + 0.20 * account_df['score_amount']
).round(4)

account_df = account_df.sort_values(
    by=['candidate_seed_score', 'account_id'],
    ascending=[False, True]
).reset_index(drop=True)

# Select top 10% as candidate seeds
num_seeds = int(len(account_df) * 0.10)
seed_accounts = account_df.head(num_seeds).copy()
seed_ids = seed_accounts['account_id'].tolist()

seed_accounts.to_csv('/content/threshold_seed_accounts.csv', index=False)

print('Weighted Seed Selection Completed')
print(f'Total Accounts          : {len(account_df)}')
print(f'Seed Accounts Selected  : {len(seed_ids)}')
seed_accounts[['account_id', 'candidate_seed_score', 'tx_frequency', 'total_tx_count']].head(10)

### Cell 4 : Eligibility Filter
Structuring requires *repeated* transactions — a single transaction can never
exhibit the pattern. Filter out seeds with fewer than `MIN_TX_COUNT` total
transactions before batching.

In [ ]:
MIN_TX_COUNT = 3   # minimum total transactions (in + out) to be eligible

eligible_seed_df = seed_accounts[seed_accounts['total_tx_count'] >= MIN_TX_COUNT].copy()
eligible_seed_ids = eligible_seed_df['account_id'].tolist()

print(f'Seed accounts before eligibility filter : {len(seed_ids)}')
print(f'Seed accounts after  eligibility filter : {len(eligible_seed_ids)}')
print(f'Filtered out (too few transactions)     : {len(seed_ids) - len(eligible_seed_ids)}')

---
## MODULE 2 — Seed Batching
---

### Cell 1 : Batch Eligible Seeds
Batching happens strictly after seed selection and eligibility filtering,
consistent with the pattern used across the other detection modules.

In [ ]:
BATCH_SIZE = 200

eligible_seed_batches = make_batches(eligible_seed_ids, batch_size=BATCH_SIZE)

print('=' * 60)
print('Seed Batch Summary')
print('=' * 60)
print(f'Eligible seeds : {len(eligible_seed_ids)}')
print(f'Batch size     : {BATCH_SIZE}')
print(f'Total batches  : {len(eligible_seed_batches)}')
print('=' * 60)

---
## MODULE 3 — Extract Full Transaction History
---

### Cell 1 : Full History Query (Incoming AND Outgoing)
Pulls every transaction touching each seed account — both as sender and as
receiver — tagged with a `direction` field. Structuring can occur on either
side (structured cash deposits coming IN, or structured withdrawals/transfers
going OUT), so both must be captured.

In [ ]:
FULL_HISTORY_QUERY = """
UNWIND $seed_ids AS origin_id

MATCH (a:Account {account_id: origin_id})

OPTIONAL MATCH (a)-[t_out:TRANSACTION]->(receiver:Account)
WITH a, collect({
    direction   : 'out',
    counterparty: receiver.account_id,
    amount      : t_out.amount_paid,
    timestamp   : t_out.timestamp,
    txn_type    : t_out.payment_format,
    laundering  : t_out.is_laundering
}) AS out_txns

OPTIONAL MATCH (sender:Account)-[t_in:TRANSACTION]->(a)
WITH a, out_txns, collect({
    direction   : 'in',
    counterparty: sender.account_id,
    amount      : t_in.amount_paid,
    timestamp   : t_in.timestamp,
    txn_type    : t_in.payment_format,
    laundering  : t_in.is_laundering
}) AS in_txns

RETURN
    a.account_id AS account_id,
    out_txns,
    in_txns
"""

print('Full transaction history query defined ✓')
print("NOTE: 'payment_format' is used as the txn_type property name based on IBM")
print("HI-Small's schema. If your property is named differently, update it above.")

### Cell 2 : Run the Full-History Sweep
Executes the query across all batches and flattens both `out_txns` and
`in_txns` into a single per-account transaction list, tagged by direction.

In [ ]:
raw_history = []

print('=' * 65)
print('Running Full Transaction History Sweep')
print('=' * 65)

start = time.time()
with driver.session() as session:
    for i, batch in enumerate(eligible_seed_batches):
        raw_history.extend(run_query_on_batch(session, FULL_HISTORY_QUERY, batch, 'history'))
        if (i + 1) % 10 == 0 or (i + 1) == len(eligible_seed_batches):
            print(f'  Batch {i+1:>4}/{len(eligible_seed_batches)} | accounts so far: {len(raw_history)}')

print(f'\nFull history sweep done in {time.time()-start:.1f}s — {len(raw_history)} accounts retrieved')


def flatten_account_history(record):
    """Merge an account's out_txns and in_txns into one chronological list
    of transactions, each tagged with its direction."""
    all_txns = (record.get('out_txns') or []) + (record.get('in_txns') or [])
    # Drop placeholder rows from OPTIONAL MATCH producing an all-null entry
    all_txns = [tx for tx in all_txns if tx.get('amount') is not None]
    return {
        'account_id': record['account_id'],
        'transactions': all_txns,
    }


account_histories = [flatten_account_history(r) for r in raw_history]
account_histories = [h for h in account_histories if h['transactions']]

print(f'Accounts with usable transaction history : {len(account_histories)}')

---
## MODULE 3A / 3B — Detection
---

### Cell 1 : Configure the Threshold Window (Fully Configurable)
This REPLACES the fixed 90–99% band. `THRESHOLD_LIMIT` is the reporting limit
being evaded (e.g. a $10,000 CTR-style threshold — adjust to whatever is
relevant for your dataset/jurisdiction). `LOWER_PCT` / `UPPER_PCT` define the
suspicious band as a percentage of that limit, and can be changed freely
without touching any downstream logic.

In [ ]:
THRESHOLD_LIMIT = 10000.0   # the reporting limit being evaded — tune per dataset
LOWER_PCT       = 0.80      # lower bound of the suspicious band, as % of limit
UPPER_PCT       = 0.99      # upper bound of the suspicious band, as % of limit

LOWER_BOUND = THRESHOLD_LIMIT * LOWER_PCT
UPPER_BOUND = THRESHOLD_LIMIT * UPPER_PCT

print(f'Threshold limit    : {THRESHOLD_LIMIT}')
print(f'Suspicious band    : {LOWER_BOUND:.2f}  –  {UPPER_BOUND:.2f}')
print(f'  ({LOWER_PCT:.0%} – {UPPER_PCT:.0%} of the limit)')
print()
print('To change the window, just edit LOWER_PCT / UPPER_PCT above —')
print('no other cell needs to change.')

### Cell 2 : Module 3A — Sub-Threshold Detector
For each account, find every transaction whose amount falls inside the
configurable `[LOWER_BOUND, UPPER_BOUND]` band, then group them by calendar
day. A day with `MIN_SUBTHRESHOLD_COUNT` or more such transactions is flagged
as a sub-threshold structuring event.

In [ ]:
MIN_SUBTHRESHOLD_COUNT = 2   # min sub-threshold txns in one day to flag


def detect_subthreshold_events(account_histories, lower=LOWER_BOUND, upper=UPPER_BOUND,
                                 min_count=MIN_SUBTHRESHOLD_COUNT):
    """
    Scan each account's transactions for amounts inside the configurable
    sub-threshold band, grouped by calendar day (UTC).
    """
    events = []

    for history in account_histories:
        account_id = history['account_id']
        txns = history['transactions']

        # Attach parsed epoch + calendar day to each transaction
        parsed = []
        for tx in txns:
            epoch = to_epoch(tx.get('timestamp'))
            if epoch is None:
                continue
            day_key = datetime.fromtimestamp(epoch, tz=timezone.utc).strftime('%Y-%m-%d')
            parsed.append({**tx, 'epoch': epoch, 'day': day_key})

        # Keep only transactions inside the configurable sub-threshold band
        in_band = [tx for tx in parsed if lower <= (tx['amount'] or 0) <= upper]

        if not in_band:
            continue

        # Group by day
        by_day = {}
        for tx in in_band:
            by_day.setdefault(tx['day'], []).append(tx)

        for day, day_txns in by_day.items():
            if len(day_txns) >= min_count:
                events.append({
                    'account_id'   : account_id,
                    'pattern_type' : 'sub_threshold',
                    'event_date'   : day,
                    'txn_count'    : len(day_txns),
                    'amounts'      : [tx['amount'] for tx in day_txns],
                    'timestamps'   : [tx['timestamp'] for tx in day_txns],
                    'txn_types'    : [tx.get('txn_type') for tx in day_txns],
                    'is_laundering': [tx['laundering'] for tx in day_txns],
                })

    return events


subthreshold_events = detect_subthreshold_events(account_histories)
print(f'Sub-threshold events detected : {len(subthreshold_events)}')

### Cell 3 : Module 3B — Frequency Burst Detector
Independent of amount — flags accounts with an unusually high count of
transactions (any amount) within a single calendar day, regardless of whether
they fall in the sub-threshold band. Catches smurfing-style bursts that don't
specifically hug the reporting limit.

In [ ]:
MIN_BURST_COUNT = 5   # min transactions in one day to flag as a frequency burst


def detect_frequency_burst_events(account_histories, min_count=MIN_BURST_COUNT):
    """Scan each account's transactions for days with an unusually high
    transaction count, independent of amount."""
    events = []

    for history in account_histories:
        account_id = history['account_id']
        txns = history['transactions']

        parsed = []
        for tx in txns:
            epoch = to_epoch(tx.get('timestamp'))
            if epoch is None:
                continue
            day_key = datetime.fromtimestamp(epoch, tz=timezone.utc).strftime('%Y-%m-%d')
            parsed.append({**tx, 'epoch': epoch, 'day': day_key})

        by_day = {}
        for tx in parsed:
            by_day.setdefault(tx['day'], []).append(tx)

        for day, day_txns in by_day.items():
            if len(day_txns) >= min_count:
                events.append({
                    'account_id'   : account_id,
                    'pattern_type' : 'frequency_burst',
                    'event_date'   : day,
                    'txn_count'    : len(day_txns),
                    'amounts'      : [tx['amount'] for tx in day_txns],
                    'timestamps'   : [tx['timestamp'] for tx in day_txns],
                    'txn_types'    : [tx.get('txn_type') for tx in day_txns],
                    'is_laundering': [tx['laundering'] for tx in day_txns],
                })

    return events


burst_events = detect_frequency_burst_events(account_histories)
print(f'Frequency burst events detected : {len(burst_events)}')

---
## MODULE 4 — Merge & Remove Duplicates
---

### Cell 1 : Merge Both Detectors & Deduplicate
Combines sub-threshold and frequency-burst events, then deduplicates using a
fingerprint of `(account_id, event_date, pattern_type)` — an account can
legitimately appear once per day per pattern type, but not twice.

In [ ]:
all_events = subthreshold_events + burst_events

if all_events:
    threshold_raw_df = pd.DataFrame(all_events)

    threshold_raw_df['fingerprint'] = (
        threshold_raw_df['account_id'].astype(str) + '::' +
        threshold_raw_df['event_date'].astype(str)  + '::' +
        threshold_raw_df['pattern_type'].astype(str)
    )

    rows_before = len(threshold_raw_df)
    threshold_raw_df = threshold_raw_df.drop_duplicates(
        subset=['fingerprint'], keep='first'
    ).reset_index(drop=True)
    rows_after = len(threshold_raw_df)

    print(f'Events before dedup : {rows_before}')
    print(f'Events after  dedup : {rows_after}')
    print(f'Duplicates removed  : {rows_before - rows_after}')
    print()
    print('Breakdown by pattern type:')
    print(threshold_raw_df['pattern_type'].value_counts().to_string())

else:
    threshold_raw_df = pd.DataFrame(columns=[
        'account_id', 'pattern_type', 'event_date', 'txn_count',
        'amounts', 'timestamps', 'txn_types', 'is_laundering', 'fingerprint'
    ])
    print('No threshold/structuring events detected.')

threshold_raw_df.head()

---
## MODULE 5 — Feature Engineering
*(`any_laundering` intentionally excluded — see note below)*
---

### Cell 1 : Define Feature Engineering Function
**`any_laundering` is NOT computed here.** The `is_laundering` list is kept
in the raw DataFrame untouched, purely so Module 7 can compute ground truth
later — but no feature derived from it enters the engineered feature set or
the risk scorer. This keeps the detector's score genuinely unsupervised,
evaluated against labels rather than partly built from them.

In [ ]:
def engineer_threshold_features(row: pd.Series) -> pd.Series:
    """
    Compute derived features for a single threshold/structuring event.
    NOTE: no laundering-label-derived feature is produced here by design.

    Threshold features  : pct_of_limit, below_limit_count, avg_gap_from_limit
    Frequency features   : txn_count_per_day (already txn_count), round_number_ratio
    Consistency features : amount_variance, cumulative_total, amount_std
    """
    amounts = [a for a in (row.get('amounts') or []) if a is not None]

    if not amounts:
        return pd.Series({
            'pct_of_limit'       : 0.0,
            'below_limit_count'  : 0,
            'avg_gap_from_limit' : 0.0,
            'round_number_ratio' : 0.0,
            'amount_variance'    : 0.0,
            'amount_std'         : 0.0,
            'cumulative_total'   : 0.0,
        })

    # ── Threshold features ───────────────────────────
    avg_amount = float(np.mean(amounts))
    pct_of_limit = avg_amount / THRESHOLD_LIMIT if THRESHOLD_LIMIT > 0 else 0.0

    below_limit_count = sum(1 for a in amounts if a < THRESHOLD_LIMIT)

    # How close, on average, transactions sit to the limit (smaller = closer = more suspicious)
    gaps = [THRESHOLD_LIMIT - a for a in amounts if a < THRESHOLD_LIMIT]
    avg_gap_from_limit = float(np.mean(gaps)) if gaps else float(THRESHOLD_LIMIT)

    # ── Frequency / round-number features ───────────────
    # Round-number amounts (multiples of 100) are a classic structuring signal —
    # deliberately chosen round figures rather than naturally-occurring amounts
    round_count = sum(1 for a in amounts if a % 100 == 0)
    round_number_ratio = round_count / len(amounts)

    # ── Consistency features ───────────────────────
    amount_variance  = float(np.var(amounts))  if len(amounts) > 1 else 0.0
    amount_std        = float(np.std(amounts))  if len(amounts) > 1 else 0.0
    cumulative_total = float(sum(amounts))

    return pd.Series({
        'pct_of_limit'       : round(pct_of_limit, 4),
        'below_limit_count'  : below_limit_count,
        'avg_gap_from_limit' : round(avg_gap_from_limit, 2),
        'round_number_ratio' : round(round_number_ratio, 4),
        'amount_variance'    : amount_variance,
        'amount_std'         : amount_std,
        'cumulative_total'   : cumulative_total,
    })


print('Feature engineering function defined ✓ (any_laundering NOT included)')

### Cell 2 : Apply Feature Engineering
Applies the function row-wise and joins the resulting columns onto the base
DataFrame. An explicit assertion confirms `any_laundering` never leaks into
the engineered feature set.

In [ ]:
if not threshold_raw_df.empty:

    print('Applying feature engineering...')

    feature_df = threshold_raw_df.apply(engineer_threshold_features, axis=1)

    enriched_df = pd.concat(
        [threshold_raw_df.reset_index(drop=True), feature_df.reset_index(drop=True)],
        axis=1
    )

    print(f'Feature engineering complete — {len(enriched_df)} events enriched')
    print(f'Total columns : {len(enriched_df.columns)}')

    assert 'any_laundering' not in feature_df.columns, 'any_laundering must not be an engineered feature'
    print('Confirmed: any_laundering is NOT in the engineered feature set ✓')

    enriched_df[[
        'account_id', 'pattern_type', 'txn_count', 'pct_of_limit',
        'below_limit_count', 'round_number_ratio', 'cumulative_total'
    ]].head()

else:
    enriched_df = threshold_raw_df.copy()
    print('No events to engineer features for.')

---
## MODULE 6 — Risk Scoring
*(reweighted — no laundering-label component)*
---

### Cell 1 : Define Risk Scoring Functions
The original 4-way weighting (threshold / frequency / consistency / label)
had its label weight (0.15) removed and the remaining three redistributed
proportionally to sum back to 1.0: **threshold 0.40, frequency 0.35,
consistency 0.25.** No component here reads `is_laundering` in any form.

In [ ]:
def compute_threshold_risk_score(row: pd.Series) -> float:
    """
    Composite risk score for one threshold/structuring event. Range: 0.0 – 1.0
    Purely unsupervised — no laundering label is read anywhere in this function.

    Dimension          Weight  Rationale
    ────────────────────────────────────────────
    threshold_score      0.40   How close amounts sit to the reporting limit
    frequency_score       0.35   How many qualifying transactions occurred that day
    consistency_score      0.25   Low variance + round numbers = deliberate structuring
    """

    # ── Threshold score: closer to the limit (smaller gap) = higher score ──
    gap = float(row.get('avg_gap_from_limit', THRESHOLD_LIMIT))
    threshold_score = max(0.0, 1.0 - (gap / THRESHOLD_LIMIT))

    # ── Frequency score: more txns in the window = higher score ────────
    # Scale: MIN_SUBTHRESHOLD_COUNT (2) → 0.0, 10+ → 1.0
    txn_count = float(row.get('txn_count', 0))
    frequency_score = min(max(txn_count - 2, 0) / 8.0, 1.0)

    # ── Consistency score: low variance + high round-number ratio = deliberate ──
    round_ratio = float(row.get('round_number_ratio', 0.0))
    variance    = float(row.get('amount_variance', 0.0))
    # Normalise variance inversely — cap comparison scale at THRESHOLD_LIMIT^2 / 4
    variance_ceiling = (THRESHOLD_LIMIT ** 2) / 4.0
    low_variance_score = max(0.0, 1.0 - min(variance / variance_ceiling, 1.0)) if variance_ceiling > 0 else 0.0
    consistency_score  = 0.5 * round_ratio + 0.5 * low_variance_score

    score = (
          0.40 * threshold_score
        + 0.35 * frequency_score
        + 0.25 * consistency_score
    )

    return round(min(score, 1.0), 4)


def assign_risk_tier(score: float) -> str:
    if   score >= 0.75: return 'CRITICAL'
    elif score >= 0.50: return 'HIGH'
    elif score >= 0.25: return 'MEDIUM'
    else:               return 'LOW'


print('Risk scoring functions defined ✓ (no laundering label used)')

### Cell 2 : Apply Scoring → Produce Final `threshold_df`
Scores every enriched event, assigns a tier, and sorts descending. Note that
`is_laundering` (the raw label list) is still present as a column — it simply
was never read by the scoring function — so Module 7 can still evaluate against it.

In [ ]:
if not enriched_df.empty:

    print('Computing risk scores...')

    enriched_df['risk_score'] = enriched_df.apply(compute_threshold_risk_score, axis=1)
    enriched_df['risk_tier']  = enriched_df['risk_score'].apply(assign_risk_tier)

    threshold_df = enriched_df.sort_values(
        by=['risk_score', 'txn_count'],
        ascending=[False, False]
    ).reset_index(drop=True)

    print(f'Risk scoring complete — {len(threshold_df)} events scored')
    print()
    print('Risk tier distribution:')
    print(threshold_df['risk_tier'].value_counts().to_string())

else:
    threshold_df = enriched_df.copy()
    threshold_df['risk_score'] = pd.Series(dtype=float)
    threshold_df['risk_tier']  = pd.Series(dtype=str)
    print('No events to score.')

threshold_df[[
    'account_id', 'pattern_type', 'event_date', 'txn_count',
    'risk_score', 'risk_tier', 'pct_of_limit'
]].head(10)

---
## MODULE 7 — Validation & Evaluation
*(`any_laundering` is computed and used HERE ONLY)*
---

### Cell 1 : Compute Ground Truth & Evaluate
This is the **only** place in the notebook where `is_laundering` is converted
into a usable label. It was carried through Modules 4–6 as raw, untouched data
specifically so it could be used here — purely for evaluating the unsupervised
risk score, never for building it.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print('=' * 65)
print('MODULE 7 — Validation & Evaluation')
print('=' * 65)

if threshold_df.empty:
    print('No events to evaluate.')

else:
    # Ground truth computed HERE ONLY — not in Module 5 or Module 6
    def compute_any_laundering(labels):
        labels_clean = [bool(l) for l in (labels or []) if l is not None]
        return int(sum(labels_clean) > 0)

    threshold_df['any_laundering'] = threshold_df['is_laundering'].apply(compute_any_laundering)

    y_true = threshold_df['any_laundering'].astype(int)

    DETECTION_THRESHOLD = 0.25
    y_pred = (threshold_df['risk_score'] >= DETECTION_THRESHOLD).astype(int)

    print(f'\nDetection threshold : {DETECTION_THRESHOLD}')
    print(f'Total events         : {len(threshold_df)}')
    print(f'Ground-truth positives (any_laundering=1) : {y_true.sum()}')
    print(f'Predicted  positives (score >= threshold) : {y_pred.sum()}')

    present_classes = sorted(y_true.unique())
    class_names     = {0: 'Clean', 1: 'Suspicious'}
    target_names    = [class_names[c] for c in present_classes]

    print('\nClassification Report:')
    print(classification_report(
        y_true, y_pred,
        labels=present_classes,
        target_names=target_names,
        zero_division=0
    ))

    print('Confusion Matrix (rows = actual, cols = predicted):')
    cm = confusion_matrix(y_true, y_pred, labels=present_classes)
    cm_df = pd.DataFrame(
        cm,
        index=  [f'Actual {n}' for n in target_names],
        columns=[f'Pred {n}'   for n in target_names]
    )
    print(cm_df.to_string())

    if y_true.nunique() > 1:
        auc = roc_auc_score(y_true, threshold_df['risk_score'])
        print(f'\nROC-AUC Score : {auc:.4f}')
    else:
        unique_class = class_names[y_true.iloc[0]]
        print(f'\nROC-AUC : skipped — all events are "{unique_class}" (only 1 class present)')

### Cell 2 : Per-Pattern-Type Breakdown
Shows whether sub-threshold or frequency-burst events catch more real
laundering — useful for tuning `LOWER_PCT`/`UPPER_PCT` and `MIN_BURST_COUNT`.

In [ ]:
if not threshold_df.empty:

    summary = threshold_df.groupby('pattern_type').agg(
        total_events      = ('risk_score',     'count'),
        avg_risk_score    = ('risk_score',     'mean'),
        laundering_events = ('any_laundering', 'sum'),
        avg_pct_of_limit  = ('pct_of_limit',    'mean'),
        avg_txn_count     = ('txn_count',       'mean'),
        critical_count    = ('risk_tier', lambda x: (x == 'CRITICAL').sum()),
        high_count        = ('risk_tier', lambda x: (x == 'HIGH').sum()),
    ).reset_index()

    summary['laundering_hit_rate'] = (
        summary['laundering_events'] / summary['total_events']
    ).round(4)

    print('Per-pattern-type summary:')
    print(summary.to_string(index=False))

    print(f'\nOverall laundering hit rate : {threshold_df["any_laundering"].mean():.2%}')
    print(f'Mean risk score             : {threshold_df["risk_score"].mean():.4f}')
    print(f'CRITICAL tier events        : {(threshold_df["risk_tier"] == "CRITICAL").sum()}')
    print(f'HIGH     tier events        : {(threshold_df["risk_tier"] == "HIGH").sum()}')

### Cell 3 : Save Outputs — Export ALL HIGH and CRITICAL Alerts
Per the requirement, this exports **every** HIGH and CRITICAL tier alert (not
a top-N sample) as a dedicated actionable file for investigators, alongside
the full results file for downstream modules.

In [ ]:
OUTPUT_DIR = '/content/drive/MyDrive/AML System/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

full_output_path  = f'{OUTPUT_DIR}/threshold_detection_results.csv'
alerts_output_path = f'{OUTPUT_DIR}/threshold_high_critical_alerts.csv'

# Full results — everything, all tiers
threshold_df.to_csv(full_output_path, index=False)

# ALL HIGH and CRITICAL alerts — no row-count cap, this is the actionable list
high_critical_alerts = threshold_df[
    threshold_df['risk_tier'].isin(['HIGH', 'CRITICAL'])
].copy()

high_critical_alerts.to_csv(alerts_output_path, index=False)

print('Outputs saved to Google Drive:')
print(f'  ✓ threshold_detection_results.csv     ({len(threshold_df)} rows — all tiers)')
print(f'  ✓ threshold_high_critical_alerts.csv  ({len(high_critical_alerts)} rows — HIGH + CRITICAL only)')
print()
print('threshold_df is ready → next: Dormancy Module')
threshold_df.head()